# Методы решения конечных игр:
## Сведение m×n игры к задаче линейного программирования и метод итераций
---

## 1. Вступление

### Актуальность темы
Теория игр – это раздел математики, изучающий рациональное поведение сторон в условиях конфликта или взаимодействия. В рамках *конечных* (матричных) игр мы рассматриваем двух игроков, каждый из которых имеет конечный набор стратегий. Когда игра с **нулевой суммой**, выигрыш одного игрока равен проигрышу другого. 

Примеров применения теории игр множество:
- Оптимизация в экономике и финансах (распределение активов с учётом худшего сценария);
- Кибербезопасность (модели "атакующий – защитник");
- Машинное обучение (adversarial learning);
- И многие другие.

Важнейшим результатом теории игр является **теорема минимакса**, гарантирующая существование оптимальных (смешанных) стратегий у обоих игроков и равенство maximin = minimax. В частности, игрок A максимально повышает свой гарантированный выигрыш, а игрок B – минимизирует тот же показатель.

### Основные понятия
- **Матричная игра**: игра задана платёжной матрицей $A$ размера $m \times n$. Элемент $a_{ij}$ означает выигрыш игрока A (и убыток B), если A выбирает стратегию $i$, B – $j$.
- **Чистая стратегия**: игрок выбирает одну конкретную строку (A) или столбец (B) с вероятностью 1.
- **Смешанная стратегия**: распределение вероятностей на чистых стратегиях. Например, $x = (x_1, ..., x_m)$ для A.
- **Цена игры** $v$: равновесная величина, равная среднему выигрышу A при условии, что оба игрока играют оптимально.
- **Седловая точка**: если она существует, обеспечивает решение в чистых стратегиях.

### План ноутбука
1. Проверка седловой точки "на бумаге" (при необходимости);
2. Формулировка игры как задачи линейного программирования (ЛП);
3. Пример с помощью `scipy.optimize.linprog`;
4. Метод итераций (метод Брауна–Робинсона): идея и практическая реализация;
5. Задачи для самостоятельной практики.

По ходу ноутбука для некоторых теоретических пунктов (помеченных **(PT)**) будут даны практические упражнения.


## 2. Седловая точка и анализ простых матриц (PT)

Иногда игра может иметь **седловую точку** $a_{ij}$ в чистых стратегиях, когда элемент матрицы является:
- Минимальным в своей строке (игрок A не может получить больше, выбирая другую стратегию, если B зафиксирован);
- Максимальным в своём столбце (игрок B не может снизить выплату, выбирая другой столбец, если A зафиксирован).

> **Практика (PT)**: Возьмём матрицу 2×2 (простую) и проверим наличие седловой точки.

    Матрица A = [ [2, 5], [4, 1] ]

1. Для строки 1 берём минимум: min(2, 5) = 2;
2. Для строки 2 берём минимум: min(4, 1) = 1;
3. Максимум из этих минимумов (maximin) = max(2, 1) = 2.

Аналогично:
- Столбец 1: max(2, 4) = 4;
- Столбец 2: max(5, 1) = 5;
Минимум из этих максимумов (minimax) = min(4, 5) = 4.

Поскольку 2 ≠ 4, седловой точки нет. Значит, оптимальное решение будет в смешанных стратегиях.

При этом, если бы мы нашли элемент $a_{ij}$, который удовлетворяет обоим условиям (минимальный по строке и максимальный по столбцу), это была бы седловая точка, и игра решалась бы в чистых стратегиях i, j.


## 3. Сведение к задаче линейного программирования

### 3.1. Постановка задачи для игрока A (прямая задача)
Пусть $x = (x_1, ..., x_m)$ – смешанная стратегия A, а $v$ – гарантированный (минимальный) выигрыш, который A стремится максимизировать. 

**Формулировка ЛП**:
- Целевая функция:  $\max v$.
- Ограничения:
  1. $\sum_{i=1}^{m} x_i = 1$,
  2. $x_i \ge 0 $ для всех i,
  3. $\sum_{i=1}^{m} a_{ij} x_i \ge v$ для всех j=1..n.

### 3.2. Постановка задачи для игрока B (двойственная)
Пусть $y = (y_1, ..., y_n)$ – стратегия B, $u$ – верхняя граница выигрыша A, которую B хочет минимизировать.

**Формулировка ЛП**:
- Целевая функция: $\min u$.
- Ограничения:
  1. $\sum_{j=1}^{n} y_j = 1$,
  2. $y_j \ge 0$ для всех j,
  3. $\sum_{j=1}^{n} a_{ij} y_j \le u$ для всех i=1..m.

В решениях обеих задач оптимальное значение совпадает и равно цене игры $v^* = u^*$. Это и есть минимаксное решение.


## 4. Пример решения игры 2×3 с помощью `scipy.optimize.linprog` (PT)

Рассмотрим в качестве демонстрации матрицу:
\[
  A = \begin{pmatrix}
  2 & 5 & 1\\
  4 & 1 & 3
  \end{pmatrix}.
\]

**Задача для игрока A**: $\max v$,
при
1. $x_1 + x_2 = 1$,
2. $x_1, x_2 \ge 0$,
3. Условие на каждый столбец $j$: $a_{1j} x_1 + a_{2j} x_2 \ge v$.

Так как `linprog` решает задачу **минимизации**, мы пишем $\min (-v)$ = $\max v$. Аналогично условию $\sum_i a_{ij} x_i \ge v$ переписываем как $\sum_i -a_{ij} x_i \le -v$.

### 4.1. Подготовка кода
Далее мы покажем код на Python. Учитывайте, что при использовании `linprog` нужно аккуратно сформировать массивы коэффициентов.


In [ ]:
import numpy as np
from scipy.optimize import linprog

# Матрица выплат для игрока A (2x3)
A = np.array([[2, 5, 1],
              [4, 1, 3]])

m, n = A.shape  # m=2, n=3

# Переменные, которые мы хотим найти:
# v, x1, x2
# Итого 3 переменные. Но для linprog представим их в виде вектора: [v, x1, x2].

# Целевая функция: minimize -v <=> maximize v
# Следовательно, c = [coefficient_v, coefficient_x1, coefficient_x2] = [-1, 0, 0]
c = [-1.0, 0.0, 0.0]

# Ограничения: sum(x_i) = 1, x_i >=0
# Для sum(x_i) = 1  представим как  x1 + x2 = 1
# В формате linprog ограничения обычно задаются либо как <=, либо >=. При равенстве используем 'A_eq' и 'b_eq'.
A_eq = [[0.0, 1.0, 1.0]]   # соответствует x1 + x2
b_eq = [1.0]              # = 1

# Для x1, x2 >= 0 мы потом зададим bounds

# Условия: a_{1j} x1 + a_{2j} x2 >= v  => -(a_{1j} x1 + a_{2j} x2) <= -v
# Переписываем в векторном виде.
# Для каждого столбца j: -A[0,j]*x1 - A[1,j]*x2 <= -v
# => v - A[0,j]*x1 - A[1,j]*x2 <= 0
# но в нашем векторе переменных = [v, x1, x2],
# => 1*v + (-A[0,j])*x1 + (-A[1,j])*x2 <= 0

A_ub = []
b_ub = []

for j in range(n):
    row = [1.0, -A[0,j], -A[1,j]]
    A_ub.append(row)
    b_ub.append(0.0)

A_ub = np.array(A_ub)
b_ub = np.array(b_ub)

# Пределы (bounds) на переменные:
# v не ограничен сверху/снизу явно, но для безопасности linprog требует задать большой интервал
# x1, x2 >= 0
bounds = [(-1e9, 1e9),  # v
          (0.0, 1e9),   # x1
          (0.0, 1e9)]   # x2

res = linprog(
    c, A_ub, b_ub, A_eq, b_eq,
    bounds=bounds,
    method='highs'  # рекомендуемый метод
)

if res.success:
    v_opt = -res.fun  # так как наша целевая была -v
    print("Оптимальное решение найдено!")
    print(f"Цена игры (v) = {v_opt:.4f}")
    x1_opt = res.x[1]
    x2_opt = res.x[2]
    print(f"Оптимальные вероятности: x1 = {x1_opt:.4f}, x2 = {x2_opt:.4f}")
else:
    print("Не удалось найти решение:", res.message)


После запуска кода мы увидим что-то вроде:
```
Оптимальное решение найдено!
Цена игры (v) = ...
Оптимальные вероятности: x1 = ..., x2 = ...
```
Соответственно, $v$ – это гарантированный выигрыш игрока A. Обычно мы (при положительной цене игры) можем ещё разделить результат на «приведённую» форму, если из матрицы предварительно вычитали или добавляли константы. Но в данном простом примере, похоже, напрямую получим средний выигрыш.


## 5. Итерационный метод (метод Брауна–Робинсона)

Метод итераций (fictitious play) даёт более «процессный» взгляд, как игроки постепенно адаптируют свои действия:
1. Инициализация: выберите чистые стратегии (например, случайно) для обоих игроков.
2. На каждом шаге игрок A видит, как часто B играл каждый столбец, и выбирает лучшую чистую стратегию против этой "средней" стратегии B. Аналогично B выбирает лучшую чистую стратегию против статистики A.
3. Обновить счётчик, разыграть очередную партию, повторять.
4. Средние частоты выбора по игроку A $\to x^*$, по игроку B $\to y^*$, а средний выигрыш $\to v^*$.

> Метод сходится к оптимальному решению при достаточном числе итераций, особенно если игра небольшого размера.

### 5.1. Пример реализации (PT)
Ниже упрощённый код, демонстрирующий идею на маленькой игре. Мы запустим несколько итераций, чтобы посмотреть, как эмпирические распределения сходятся к равновесию.


In [ ]:
import random
import numpy as np

def best_response_A(A, freqB):
    """
    Игрок A: по вероятностям freqB (статистике, как B играет столбцы)
    ищет чистую стратегию i*, которая максимизирует средний выигрыш.
    Возвращает индекс i*.
    """
    m, n = A.shape
    best_i = 0
    best_val = -1e9
    for i in range(m):
        # Вычислить средний выигрыш, если A выбирает i
        payoff_i = sum(A[i, j] * freqB[j] for j in range(n))
        if payoff_i > best_val:
            best_val = payoff_i
            best_i = i
    return best_i

def best_response_B(A, freqA):
    """
    Игрок B: по вероятностям freqA (статистике, как A играет строки)
    ищет чистую стратегию j*, которая минимизирует средний выигрыш A.
    Возвращает индекс j*.
    """
    m, n = A.shape
    best_j = 0
    best_val = 1e9
    for j in range(n):
        # Вычислить средний выигрыш, если B выбирает j
        payoff_j = sum(A[i, j] * freqA[i] for i in range(m))
        if payoff_j < best_val:
            best_val = payoff_j
            best_j = j
    return best_j

# Попробуем метод итераций на той же матрице 2x3
A_matrix = np.array([[2, 5, 1],
                     [4, 1, 3]])
m, n = A_matrix.shape

# Частоты выбора строк A:
countA = np.zeros(m)
# Частоты выбора столбцов B:
countB = np.zeros(n)

# Начнём с произвольных чистых стратегий
current_i = 0  # A выбирает строку 0
current_j = 0  # B выбирает столбец 0

num_iterations = 10

for step in range(1, num_iterations+1):
    # Обновим счётчики
    countA[current_i] += 1
    countB[current_j] += 1

    # Эмпирические вероятности
    freqA = countA / step
    freqB = countB / step

    # Найдём лучшую чистую стратегию для A, если B играет freqB
    current_i = best_response_A(A_matrix, freqB)
    # Найдём лучшую чистую стратегию для B, если A играет freqA
    current_j = best_response_B(A_matrix, freqA)

    # Посмотрим, как изменяются частоты выбора
    if step % 2 == 0:  # печатать каждые 2 итерации
        print(f"Итерация {step}: freqA={freqA}, freqB={freqB}")

print("\nИтоговые эмпирические вероятности A:", countA / num_iterations)
print("Итоговые эмпирические вероятности B:", countB / num_iterations)


Видно, что по мере итераций, распределения (частоты) будут постепенно "подтягиваться" к некоторому балансу. 
Это **не** гарантирует монотонное приближение, но в среднем (при большом количестве шагов) в пределе мы получим оптимальные $x^*, y^*$.

При реальном использовании метода можно увеличить `num_iterations` до сотен, тысяч и следить за сходимостью. 
Этот метод хорош, если мы хотим моделировать пошаговое взаимодействие или не можем заранее знать стратегию противника.


## 6. Задачи для самостоятельной практики

### 6.1. Базовые задания (на бумаге)
1. Седловая точка: дана матрица $2\times2$ ($[ [3, 4], [2, 5] ]$). Найти:
   - Минимальные элементы строк;
   - Максимальные элементы столбцов;
   - Проверить, есть ли седловая точка.

2. Преобразовать матрицу $2\times2$ ($[ [2, 5], [4, 1] ]$) в систему неравенств для игрока A. Выписать явные ограничения в форме $x_1 + x_2 = 1$, $x_i\ge0$, $2x_1 + 4x_2\ge v$ и т.д. 

### 6.2. Средние задания (в ноутбуке с Python)
1. **Работа с `linprog`**. Сгенерируйте случайную матрицу размером $3\times3$ (элементы в диапазоне от 0 до 10), решите задачу для игрока A по аналогии с кодом выше. 
   - Определите оптимальные вероятности $x^*$ и цену игры.
   - Проверьте ("подставьте") результат: действительно ли для любого столбца $\sum_i a_{ij} x_i \ge v$?


2. **Итерационный метод**. Реализуйте метод Брауна–Робинсона для $3\times3$ матрицы. Повторите процесс на 50 итерациях. Отследите, к каким распределениям $freqA, freqB$ сходится метод. Сравните результат с решением ЛП.



### 6.3. Продвинутые задания (реальные кейсы)
1. **Оптимизация портфеля против "природы"**. Допустим, у вас есть 3 актива, и "природа" может выбрать 1 из 3 сценариев (столбцы) с разными доходностями. Постройте матрицу выигрышей и решите задачу:$\max v$ (где $v$ – доход в самом плохом сценарии). Найдите распределение, которое гарантирует наибольший минимальный доход.


2. **Adversarial ML**. Упрощённо задайте 2 варианта обучения модели (допустим, 2 разных гиперпараметра) и 3 типа атакующих входов (столбцы). Оцените процент ошибок в каждом случае, сформируйте матрицу как "-(процент ошибок)", чтобы это выглядело как выигрыш модели. Решите задачу, найдите $x^*$ – смешанную стратегию (вероятности выбора гиперпараметра). 



Все эти задания помогают отработать навыки:
- Правильно формулировать задачу в терминах ЛП;
- Писать и проверять код c помощью `scipy.optimize`;
- Понимать логику итерационных методов и их связь с распределениями стратегий.

## 7. Заключение
1. Мы рассмотрели два ключевых подхода к решению конечных игр:
   - Приведение игры к задаче линейного программирования (и использование решателя типа `scipy.optimize.linprog`);
   - Итерационный метод (метод Брауна–Робинсона) для пошагового поиска равновесия.
2. Поняли, что при наличии седловой точки задача упрощается до чистых стратегий.
3. Важность темы:
   - Применение в финансах, безопасности, машинном обучении;
   - Навык формализации конфликтной ситуации и постановки задачи ЛП – ценный для рынка труда.

Надеюсь, этот материал позволит вам не только понять основы матричных игр и методов решения, но и применить их на практике.

> **Дополнительно**: дальнейшее развитие включает изучение некооперативных игр Нэша, многопользовательских игр, а также продвинутых алгоритмов типа no-regret learning или методов случайного градиентного спуска/подъёма.
